In [1]:
from octo.model.octo_model import OctoModel
import jax
from PIL import Image
import numpy as np
import jax.numpy as jnp
import time
from os import listdir
from os.path import isfile, join

2025-04-26 03:14:38.156992: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-26 03:14:38.157014: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-26 03:14:38.157557: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-26 03:14:38.496616: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/home/hyperdog/anaconda3/envs/octo/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook i

In [2]:
# Load model
model = OctoModel.load_pretrained("/home/hyperdog/octo/checkpoints/shampoo_100_no_force/", 1)
data_path = "/home/hyperdog/rlds_dataset_builder/grasping_dataset/data/pick_shampoo_100/"
onlyfiles = [f for f in listdir(data_path) if isfile(join(data_path, f))]
file = onlyfiles[10]
data = np.load(data_path + file, allow_pickle=True)

In [7]:
# Load the image
i = 50
image1 = jnp.array(Image.fromarray(data[i]['image_main']).resize((256, 256), Image.Resampling.LANCZOS))
image2 = jnp.array(Image.fromarray(data[i-1]['image_main']).resize((256, 256), Image.Resampling.LANCZOS))
image3 = jnp.array(Image.fromarray(data[i]['image_wrist']).resize((128, 128), Image.Resampling.LANCZOS))
image4 = jnp.array(Image.fromarray(data[i-1]['image_wrist']).resize((128, 128), Image.Resampling.LANCZOS))
# image_stacked = jnp.stack([image1, image2], axis=0)
image_primary = jnp.stack([image2, image1], axis=0)
image_primary = image_primary[jnp.newaxis, ...]
image_wrist = jnp.stack([image4, image3], axis=0)
image_wrist = image_wrist[jnp.newaxis, ...]
state = jnp.stack([jnp.array(data[i-1]['state'][-2]), jnp.array(data[i]['state'][-2])], axis=0) 
state = state[jnp.newaxis, ...]
observation = {"image_primary": image_primary, 
               "image_wrist": image_wrist, 
               "proprio": state[..., None],
               "timestep_pad_mask": jnp.array([[True, True]]),
               "timestep": jnp.array([[0, 1]]),
                "pad_mask_dict": {
                    "timestep": jnp.array([[True, True]]),  # Shape: (1, 2)
                    "image_primary": jnp.array([[True, True]]),  # Shape: (1, 2)
                    "image_wrist": jnp.array([[True, True]]),
                    "proprio": jnp.array([[True, True]])
                },
                "task_completed": jnp.zeros((1, 2, 4))}  # Shape: (1, 2, 4)
# image_stacked.shape

In [8]:
task_texts = ["Pick up the gray cylinder."]
task = model.create_tasks(texts=task_texts)
action = model.sample_actions(observation, task, rng=jax.random.PRNGKey(0),
                              unnormalization_statistics=model.dataset_statistics["action"],)[0][0]

In [25]:
true_action = np.append(data[i]['action'][8:], data[i]['action'][6])
true_action

array([ 0.0761143 , -0.01554841, -0.06424403, -0.05667162,  0.01469648,
        0.05230703,  1.        ], dtype=float32)

In [26]:
action

Array([ 0.07519681, -0.01597331, -0.06240426, -0.0593746 ,  0.01395564,
        0.05172981,  1.0065527 ], dtype=float32)

In [17]:
sum((true_action[:-1] - action[:-1])**2)

Array(1.8618406e-05, dtype=float32)

In [ ]:
data[i]['image_main'][...,:-1].astype(np.uint8)

(480, 640, 3)

In [17]:
task_texts = [data[0]['language_instruction']]
task = model.create_tasks(texts=task_texts)
error = 0
start = time.time()
# gripper_pose = []
for i in range(1, len(data)):
    image1 = jnp.array(Image.fromarray(data[i]['image_main'][...,:-1].astype(np.uint8)).resize((256, 256), Image.Resampling.LANCZOS))
    image2 = jnp.array(Image.fromarray(data[i-1]['image_main'][...,:-1].astype(np.uint8)).resize((256, 256), Image.Resampling.LANCZOS))
    image3 = jnp.array(Image.fromarray(data[i]['image_wrist']).resize((128, 128), Image.Resampling.LANCZOS))
    image4 = jnp.array(Image.fromarray(data[i-1]['image_wrist']).resize((128, 128), Image.Resampling.LANCZOS))
    image_primary = jnp.stack([image2, image1], axis=0)
    image_primary = image_primary[jnp.newaxis, ...]
    image_wrist = jnp.stack([image4, image3], axis=0)
    image_wrist = image_wrist[jnp.newaxis, ...]
    state = jnp.stack([jnp.array(data[i-1]['state'][-2:]), jnp.array(data[i]['state'][-2:])], axis=0) 
    state = state[jnp.newaxis, ...]
    observation = {"image_primary": image_primary, 
                   "image_wrist": image_wrist,
                   "proprio": state,
                "timestep_pad_mask": jnp.array([[True, True]]),
                "timestep": jnp.array([[0, 1]]),
                    "pad_mask_dict": {
                        "timestep": jnp.array([[True, True]]),  # Shape: (1, 2)
                        "image_primary": jnp.array([[True, True]]),  # Shape: (1, 2)
                        "image_wrist": jnp.array([[True, True]]),
                        "proprio": jnp.array([[True, True]])
                    },
                    "task_completed": jnp.zeros((1, 2, 4))}  # Shape: (1, 2, 4)
    action = model.sample_actions(observation, task, rng=jax.random.PRNGKey(0),
                              unnormalization_statistics=model.dataset_statistics["action"],)[0][0]
    # gripper_pose.append(round(action[6]))
    true_action = data[i]['action']
    # true_action = np.append(data[i]['action'][8:], data[i]['action'][6])

    mse = sum((true_action - action)**2)
    error+=mse
print((time.time() - start)/(len(data)-1))
print(error)

0.04046237712003747
0.028254142
